In [1]:

import influxdb_client, os, time
from influxdb_client import InfluxDBClient, Point, WritePrecision
from influxdb_client.client.write_api import SYNCHRONOUS
import pandas as pd
from datetime import datetime, timedelta, timezone


# NOT SECURITY FRIENDLY LOL
INFLUXDB_TOKEN="5hsm8F75hKkY0TBmlHuaoDES42TlaOdDbVVCnLLDKFsKvlCgLhyZLgbYMF_4y1M1aB9N-kPa9VLEND7f43zgIw=="

token = os.environ.get("INFLUXDB_TOKEN")
org = "EST"
url = "http://localhost:8086"

client = influxdb_client.InfluxDBClient(url=url, token=INFLUXDB_TOKEN, org=org)


In [2]:


df = pd.read_csv("results/csv/simulation_data.csv", delimiter=',') # all the data
traj_df = pd.read_csv("results/csv/trajectory_coords.csv", delimiter=',') # the trajectory data

# we artificially add a timestamp to the data to vbv convert to datetime
now = datetime(2025, 1, 1, 0, 0, 0)
df['times_telecom'] = df['times_telecom'].apply(lambda x: now + timedelta(seconds=x))

# the full df
df = pd.concat([df.reset_index(drop=True), traj_df.reset_index(drop=True)], axis=1)
print(df.keys())
print(df.head())


Index(['times_telecom', 'visibility', 'data', 'data_GNSS_TOF', 'data_HK',
       'times_eps', 'battery', 'consumption', 'generation', 'eclipse',
       'solar_cells_efficiency', 'times_modes', 'modes', 'times_altitude',
       'altitude', 'times_orbital', 'RAAN', 'AOP', 'ECC', 'INC',
       'times_eclipse', 'times_density', 'density', 'latitude_deg',
       'longitude_deg'],
      dtype='object')
        times_telecom  visibility      data  data_GNSS_TOF   data_HK  \
0 2025-01-01 00:00:00         0.0  0.000000          0.000  0.000000   
1 2025-01-01 00:00:10         0.0  0.065667          0.065  0.000667   
2 2025-01-01 00:00:20         0.0  0.131334          0.130  0.001334   
3 2025-01-01 00:00:30         0.0  0.197001          0.195  0.002001   
4 2025-01-01 00:00:40         0.0  0.262668          0.260  0.002668   

   times_eps   battery  consumption  generation  eclipse  ...  times_orbital  \
0        0.0  274752.0         0.00         0.0      1.0  ...            0.0   
1      

In [3]:
# Write data to InfluxDB
bucket="NICE"
write_api = client.write_api(write_options=SYNCHRONOUS)
delete_api = client.delete_api()

# Delete all previous data from bucket
start = "1970-01-01T00:00:00Z"
stop = datetime.now(timezone.utc).isoformat()
delete_api.delete(start, stop, '', bucket=bucket, org=org)



# the subsystems' times are not included because for now they are the same for all
# the .tag are used to index the data, and can be used for filtering


# Pre-build list of points
points = [
    Point("satellite_data")
        .tag("mode", int(row["modes"]))
        .tag("visible", int(row["visibility"]))
        .field("visibility", float(row["visibility"]))
        .field("data", float(row["data"]))
        .field("data_GNSS_TOF", float(row["data_GNSS_TOF"]))
        .field("data_HK", float(row["data_HK"]))
        .field("battery", float(row["battery"]))
        .field("consumption", float(row["consumption"]))
        .field("generation", float(row["generation"]))
        .field("eclipse", float(row["eclipse"]))
        .field("modes", float(row["modes"]))
        .field("altitude", float(row["altitude"]))
        .field("RAAN", float(row["RAAN"]))
        .field("AOP", float(row["AOP"]))
        .field("ECC", float(row["ECC"]))
        .field("INC", float(row["INC"]))
        .field("density", float(row["density"]))
        .field("Lat", float(row["latitude_deg"]))
        .field("Lng", float(row["longitude_deg"]))
        .field("solar_cells_efficiency", float(row["solar_cells_efficiency"]))
        .time(row["times_telecom"], write_precision=WritePrecision.NS)
    for _, row in df.iterrows()
]

# Send all points at once
write_api.write(bucket=bucket, org=org, record=points)

print("Upload complete.")





Upload complete.


In [4]:
# to test some queries
query_api = client.query_api()

query = """
from(bucket: "NICE")
  |> range(start: 0)
  |> filter(fn: (r) => r._measurement == "satellite_data")
  |> filter(fn: (r) => r._field == "latitude")
"""

tables = query_api.query(query, org="EST")
results = []
for table in tables:
    for record in table.records:
        results.append((record.get_field(),record.get_value(), record.get_time(), record.get_start(), record.get_stop()))

max_val = 5
for i in range(max_val):
    print(results[i])

IndexError: list index out of range